## Import

In [31]:
import random
import pandas as pd
import numpy as np
import os
import re
import glob
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2
import torchvision.models as models

from tqdm import tqdm

import warnings
warnings.filterwarnings(action='ignore') 

In [32]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

## Hyperparameter Setting

In [33]:
CFG = {
    'IMG_SIZE':224,
    'EPOCHS':5,
    'LEARNING_RATE':1e-3,
    'BATCH_SIZE':32,
    'SEED':777
}

## Fixed RandomSeed

In [34]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(CFG['SEED'])

## Data Pre-processing

In [35]:
df = pd.read_csv('./train.csv')

In [36]:
train_len = int(len(df) * 0.8)
train_df = df.iloc[:train_len]

remaining_df = df.iloc[train_len:]
val_len = int(len(remaining_df) * 0.5)

val_df = remaining_df.iloc[:val_len]
test_df = remaining_df.iloc[val_len:]

In [37]:
train_label_vec = train_df.iloc[:, 2:].values.astype(np.float32)
val_label_vec = val_df.iloc[:, 2:].values.astype(np.float32)
test_label_vec = test_df.iloc[:, 2:].values.astype(np.float32)

In [38]:
CFG['label_size'] = train_label_vec.shape[1]

In [39]:
CFG['label_size']

3467

## CustomDataset

In [40]:
class CustomDataset(Dataset):
    def __init__(self, img_path_list, label_list, transforms=None):
        self.img_path_list = img_path_list
        self.label_list = label_list
        self.transforms = transforms
        
    def __getitem__(self, index):
        img_path = self.img_path_list[index]
        
        image = cv2.imread(img_path)
        
        if self.transforms is not None:
            image = self.transforms(image=image)['image']
        
        if self.label_list is not None:
            label = self.label_list[index]
            return image, label
        else:
            return image
        
    def __len__(self):
        return len(self.img_path_list)

In [41]:
train_transform = A.Compose([
                            A.Resize(CFG['IMG_SIZE'],CFG['IMG_SIZE']),
                            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0, always_apply=False, p=1.0),
                            ToTensorV2()
                            ])

test_transform = A.Compose([
                            A.Resize(CFG['IMG_SIZE'],CFG['IMG_SIZE']),
                            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0, always_apply=False, p=1.0),
                            ToTensorV2()
                            ])

In [42]:
train_dataset = CustomDataset(train_df['path'].values, train_label_vec, train_transform)
train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True, num_workers=0)

val_dataset = CustomDataset(val_df['path'].values, val_label_vec, test_transform)
val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False, num_workers=0)

test_dataset = CustomDataset(test_df['path'].values, None, test_transform)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False, num_workers=0)

## Model Define

In [43]:
class MBConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, expand_ratio):
        super(MBConvBlock, self).__init__()
        hidden_dim = in_channels * expand_ratio
        self.expand = in_channels != out_channels
        self.block = nn.Sequential(

            nn.Conv2d(in_channels, hidden_dim, 1, 1, 0, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),
            

            nn.Conv2d(hidden_dim, hidden_dim, kernel_size, stride, kernel_size//2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),
            
            nn.Conv2d(hidden_dim, out_channels, 1, 1, 0, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        
    def forward(self, x):
        if self.expand:
            return self.block(x)
        else:
            return x + self.block(x)

In [44]:
class EfficientNet_base(nn.Module):
    def __init__(self):
        super(EfficientNet_base, self).__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )
        
        self.blocks = nn.Sequential(
            MBConvBlock(32, 16, 3, 1, 1),
            MBConvBlock(16, 24, 3, 2, 6),
            MBConvBlock(24, 40, 5, 2, 6),
            MBConvBlock(40, 80, 3, 2, 6),
            MBConvBlock(80, 112, 5, 1, 6),
            MBConvBlock(112, 192, 5, 2, 6),
            MBConvBlock(192, 320, 3, 1, 6)
        )
        
        self.head = nn.Sequential(
            nn.Conv2d(320, 1280, 1, 1, 0, bias=False),
            nn.BatchNorm2d(1280),
            nn.ReLU6(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(1280, 1000)
        )
        
    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        return x

In [45]:
class EfficientNet(nn.Module):
    def __init__(self, gene_size=CFG['label_size']):
        super(EfficientNet, self).__init__()
        self.backbone = EfficientNet_base()
        self.regressor = nn.Linear(1000, gene_size)
        
    def forward(self, x):
        x = self.backbone(x)
        x = self.regressor(x)
        return x

## Train

In [46]:
def train(model, optimizer, train_loader, val_loader, scheduler, device):
    model.to(device)
    criterion = nn.MSELoss().to(device)
    
    best_loss = float('inf')
    best_model_wts = model.state_dict()
    
    for epoch in range(1, CFG['EPOCHS']+1):
        model.train()
        train_loss = []
        for imgs, labels in tqdm(iter(train_loader)):
            imgs = imgs.float().to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            output = model(imgs)
            loss = criterion(output, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss.append(loss.item())
                    
        _val_loss = validation(model, criterion, val_loader, device)
        _train_loss = np.mean(train_loss)
        print(f'Epoch [{epoch}], Train Loss : [{_train_loss:.5f}] Val Loss : [{_val_loss:.5f}]')
       
        if scheduler is not None:
            scheduler.step(_val_loss)
            
        if _val_loss < best_loss:
            best_loss = _val_loss
            best_model_wts = model.state_dict()
            
    model.load_state_dict(best_model_wts)
    
    return model


In [47]:
def validation(model, criterion, val_loader, device):
    model.eval()
    val_loss = []

    with torch.no_grad():
        for imgs, labels in tqdm(iter(val_loader)):
            imgs = imgs.float().to(device)
            labels = labels.to(device)
            
            pred = model(imgs)
            
            loss = criterion(pred, labels)
            
            val_loss.append(loss.item())
        
        _val_loss = np.mean(val_loss)
    
    return _val_loss

In [48]:
def test(model, criterion, test_loader, device):
    model.eval()
    test_loss = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in tqdm(iter(test_loader)):
            labels = labels.to(device)

            pred = model(imgs)

            loss = criterion(pred, labels)
            test_loss.append(loss.item())

            all_preds.append(pred.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
        _test_loss = np.mean(test_loss)

    print(f'Test Loss : [{_test_loss:.5f}]')
    return _test_loss, all_preds, all_labels

## Run!!

## Ensemble

In [ ]:
model1 = EfficientNet()
optimizer1 = torch.optim.Adam(params=model1.parameters(), lr=CFG["LEARNING_RATE"])
scheduler1 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer1, mode='min', factor=0.5, patience=2, threshold_mode='abs', min_lr=1e-8, verbose=True)
trained_model1 = train(model1, optimizer1, train_loader, val_loader, scheduler1, device)

model2 = EfficientNet()
optimizer2 = torch.optim.Adam(params=model2.parameters(), lr=CFG["LEARNING_RATE"] * 0.8)  
scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode='min', factor=0.5, patience=2, threshold_mode='abs', min_lr=1e-8, verbose=True)
trained_model2 = train(model2, optimizer2, train_loader, val_loader, scheduler2, device)

model3 = EfficientNet()
optimizer3 = torch.optim.Adam(params=model3.parameters(), lr=CFG["LEARNING_RATE"] * 1.2) 
scheduler3 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer3, mode='min', factor=0.5, patience=2, threshold_mode='abs', min_lr=1e-8, verbose=True)
trained_model3 = train(model3, optimizer3, train_loader, val_loader, scheduler3, device)

## Inference

In [ ]:
test = pd.read_csv('./test.csv')

In [ ]:
test_dataset = CustomDataset(test['path'].values, None, test_transform)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False, num_workers=0)

In [ ]:
def inference(model, test_loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for imgs in tqdm(test_loader):
            imgs = imgs.to(device).float()
            pred = model(imgs)
            
            preds.append(pred.detach().cpu())
    
    preds = torch.cat(preds).numpy()

    return preds

In [ ]:
criterion = nn.MSELoss()

val_loss1 = validation(trained_model1, criterion, val_loader, device)
val_loss2 = validation(trained_model2, criterion, val_loader, device)
val_loss3 = validation(trained_model3, criterion, val_loader, device)

losses = [val_loss1, val_loss2, val_loss3]
inverse_losses = [1/loss for loss in losses] 

total_inverse_loss = sum(inverse_losses)
weights = [loss / total_inverse_loss for loss in inverse_losses]

print(f"Calculated weights based on validation losses: {weights}")

preds1 = inference(trained_model1, test_loader, device)
preds2 = inference(trained_model2, test_loader, device)
preds3 = inference(trained_model3, test_loader, device)

ensemble_preds = (preds1 * weights[0] + preds2 * weights[1] + preds3 * weights[2])

submit = pd.read_csv('./sample_submission.csv')
submit.iloc[:, 1:] = ensemble_preds.astype(np.float32)
submit.to_csv('./efficientnet_weighted_ensemble_submit.csv', index=False)

print(f"Weighted Ensemble Predictions saved to efficientnet_weighted_ensemble_submit.csv")

## Submission

In [ ]:
submit = pd.read_csv('./sample_submission.csv')
submit.iloc[:, 1:] = np.array(preds).astype(np.float32)
submit.to_csv('./baseline_submit.csv', index=False)